# Оценка вычислительной сложности алгоритма. Сортировка данных. Индексы и хеши

In [ ]:
import random
from matplotlib import pyplot as plt
import numpy as np
import math
from copy import copy
from collections import defaultdict

random.seed(10)

## Самостоятельная работа

Вопросы:
- Что такое безмасштабные сети? Какое у них есть основное свойство? Объясните своими словами, как алгоритм генерации Барабаши-Альберта способствует выполнению этого свойства;
- Почему наличие отрицательных циклов и даже просто отрицательных ребер в графе делает алгоритм Дейкстры неприменимым?


## Вычислительная сложность алгоритма

На предыдущих парах мы несколько раз говорили о каких-то алгоритмах как о более или менее тяжелых и дорогих. Чаще всего это выражается в количестве простых шагов, требующихся для выполнения поставленной задачи. Но обычно нас больше интересует, как будет расти это количество с увеличением объема данных.

Например, искать все пути в графе на 10 вершин не так уж и долго. А вот в графе на миллион вершин - значительно дольше. Для точного обозначения этой разницы используется Big O нотация, описывающая, по какому закону будет расти вычислительная сложность при увеличении размера массива.

Важно, что чаще всего Big O используют для оценки __худшего случая__, который не обязательно произойдет с вами при конкретном применении. Часто мы еще хотим понимать, как алгоритм будет работать __в лучшем случае__ и __в среднем__.

Разберем на примере поиска элемента в массиве.

**У нас есть массив из N элементов (пусть для простоты это будут целые числа). За какое количество шагов мы сможем проверить, содержится ли число x в этом массиве?**

Строго говоря, для ответа на этот вопрос нам не хватает данных:
- Упорядочен ли массив?
- Какой алгоритм поиска мы планируем применять?

Обсудим сначала самый простой вариант - линейный поиск. В рамках этого алгоритма мы просто перебираем элементы в том порядке, в котором они храняется в массиве, и каждый сравниваем с искомым числом. Если мы дошли до конца массива, а число так и не было найдено, то его в массиве нет.

Попробуем реализовать такой алгоритм и посмотреть, как кол-во шагов будет зависеть от кол-ва объектов в массиве:

In [ ]:
def linear_search(lst, x):
    # на каждом шаге будет увеличивать счетчик шагов
    n_steps = 0
    for item in lst:
        n_steps += 1
        if item == x:
            return item, n_steps
    return None, n_steps

Создадим массив

In [ ]:
N = 30
rng = (0, 100)
lst = [random.randint(*rng) for _ in range(N)]
print(lst)

Попробуем найти в нем существующий и не существующий элемент

In [ ]:
linear_search(lst, lst[10]), linear_search(lst, 123)

А теперь посмотрим, сколько шагов будет делаться для разных длин массивов при условии худшего случая - поиска последнего элемента:

In [ ]:
stat = []
for N in range(10, 100):
    lst = list(range(0, N))
    random.shuffle(lst)
    _, n = linear_search(lst, lst[-1])
    stat.append([N, n])
stat = np.array(stat)

plt.scatter(stat[:, 0], stat[:, 1])
plt.xlabel('Array size')
plt.ylabel('N steps');

Теперь мы наглядно видим, почему алгоритм называется __линейным__ поиском - кол-во шагов, в худшем случае линейно зависит от кол-ва элементов в массиве.

С точки зрения Big O нотации это будет алгоритм вычислительной сложности $O(n)$.

Теперь предположим, что наш массив уже отсортирован (например, по возрастанию). Тогда линейный поиск применять странно, так как он никак не учитывает порядок элементов.

Здесь нам на помощь приходит бинарный поиск. Для поиска элемента x в массиве из N элементов он будет работать так:
1. Берем элемент на позиции N // 2;
2. Сравниваем его с x. Если x больше, то сдвигаем границы поиска на часть справа от текущего элемента, если меньше - на часть слева;
3. Повторяем пункты 1-2 для нового элемента, пока не найдем искомое или не получим пустые массивы для поиска.

Посмотрим на практике:

In [ ]:
def binary_search(lst, x):
    n_steps = 0

    left = 0
    right = len(lst) - 1

    while left <= right:
        n_steps += 1

        mid = (left + right) // 2

        if lst[mid] == x:
            return lst[mid], n_steps
        elif lst[mid] < x:
            left = mid + 1
        else:
            right = mid - 1

    return None, n_steps

Применение

In [ ]:
n_values = np.arange(10, 10000)
steps = []

for n in n_values:
    lst = list(range(n))  # отсортированный массив
    x = -1                # ищем элемент, которого нет, это худший случай

    _, n_steps = binary_search(lst, x)
    steps.append(n_steps)

plt.figure(figsize=(8, 5))
plt.plot(n_values, steps)

plt.xlabel("Array size")
plt.ylabel("N steps");

Видим, что это очень похоже на график логарифмической функции. И правда, если посчитать, сколько раз можно поделить массив длины N пополам, то можно обнаружить, что это $log_2(n)$.

В Big O нотации это $O(log(n))$. И это сильно лучше, чем линейный поиск, особенно для больших массивов.

Еще обычно выделяют алгоритмы сложности:
- $O(1)$ - константные алгоритмы; например, взятие элемента из списка по индексу;
- $O(n * log(n))$ - сюда относятся многие эффективные алгоритмы сортировки, мы их обсудим через пару минут;
- $O(n^2)$ - квадратичная сложность; например, двойной вложенный цикл;
- $O(n^3)$ - тройной вложенный цикл;
- $O(n!)$ - чаще всего это перебор всех возможных комбинаций из элементов массива (кошмар, этого надо избегать всеми силами!)


Посмотрим на графики этих функций рядом (осторожно, логарифмическая шкала):


In [ ]:
# факториал быстро взрывается, так что только до 20
n_values = np.arange(2, 20)
factorial = np.array([math.factorial(n) for n in n_values])

plt.figure(figsize=(10, 6))

plt.plot(n_values, np.ones_like(n_values), label="O(1)")
plt.plot(n_values, np.log2(n_values), label="O(log n)")
plt.plot(n_values, n_values, label="O(n)")
plt.plot(n_values, n_values * np.log2(n_values), label="O(n log n)")
plt.plot(n_values, n_values ** 2, label="O(n^2)")
plt.plot(n_values, n_values ** 3, label="O(n^3)")
plt.plot(n_values, factorial, label="O(n!)")

plt.xlabel("N")
plt.ylabel("N steps")

# иначе из-за факториала мы ничего не увидим
plt.yscale("log")
plt.grid(True)
plt.legend()

plt.show()

## Алгоритмы сортировки

### Алгоритмы сортировки квадратичной сложности

Все алгоритмы здесь можно легко представить при помощи "камушков и перышек". И даже [станцевать](https://www.youtube.com/watch?v=WuGvUFvG7yo).


#### Сортировка пузырьком

**Шаг 1.** Для $i\in[1, N]$ повторить Шаг 2 для элементов в интервале [0, N-i].

**Шаг 2.** Возьмем очередной элемент и его соседа справа. Если они находятся не в нужном порядке, переставим их местами. Перейдем к следующему по счету элементу.

В результате выполнения Шага 2 самый большой (или самый маленький) элемент будет вытолкнут на самую правую позицию. Сам шаг имеет линейную сложность. Теперь нам необходимо повторить Шаг 2 столько раз, сколько в списке есть элементов.

In [ ]:
def bubble_sort(a):
    n_steps = 0
    N = len(a)

    # это чтобы не менять изначальный массив
    a = copy(a)

    for i in range(N - 1):
        for j in range(N - i - 1):
            n_steps += 1  # сравнение

            if a[j] > a[j + 1]:
                a[j], a[j + 1] = a[j + 1], a[j]
                n_steps += 1  # перестановка

    return a, n_steps

In [ ]:
N = 10000
lst = [random.randint(0, N) for i in range(N)]

In [ ]:
lst, n = bubble_sort(lst)
print(n)
print(lst[10:30])

Теперь посмотрим на зависимость от длины:

In [ ]:
n_values = np.arange(10, 100) # иначе очент долго
steps = []

for n in n_values:
    lst = [random.randint(0, n) for i in range(n)]

    _, n_steps = bubble_sort(lst)
    steps.append(n_steps)

plt.figure(figsize=(4, 6))
plt.plot(n_values, steps)

plt.xlabel("Array size")
plt.ylabel("N steps");

#### Сортировка вставкой

Теперь представим себе, что к нам последовательно приходят значения и нам надо добавлять их в список, поддерживая его отсортированность. Тогда мы можем применить алгоритм бинарного поиска, который подскажет нам на какую позицию надо вставлять значение, и формировать новый массив, со вставленным элементом.

В итоге мы получаем сортировку вставкой, которая работает по следующему принципу:
1. Считаем, что первый элемент уже отсортирован;
2. Берём следующий элемент и вставляем его в правильную позицию среди уже отсортированных элементов;
3. Для этого сдвигаем все элементы, которые больше текущего, вправо;
4. Повторяем шаги 2-3, пока не пройдём весь массив.

Новые элементы будут добавляться через шаги 2-3.

In [ ]:
def insertion_sort(a):
    n_steps = 0
    N = len(a)
    a = copy(a)

    for i in range(1, N):
        key = a[i]
        j = i - 1

        # сдвигаем элементы вправо, если они больше key
        while j >= 0:
            n_steps += 1  # сравнение

            if a[j] > key:
                a[j + 1] = a[j]
                n_steps += 1  # сдвиг
                j -= 1
            else:
                break

        a[j + 1] = key

    return a, n_steps

Потестируем

In [ ]:
N = 10000
lst = [random.randint(0, N) for i in range(N)]

In [ ]:
lst, n = insertion_sort(lst)
print(n)
print(lst[10:30])

Теперь посмотрим на зависимость от длины:

In [ ]:
n_values = np.arange(10, 100) # иначе очент долго
steps = []

for n in n_values:
    lst = [random.randint(0, n) for i in range(n)]

    _, n_steps = insertion_sort(lst)
    steps.append(n_steps)

plt.figure(figsize=(4, 6))
plt.plot(n_values, steps)

plt.xlabel("Array size")
plt.ylabel("N steps");

С одной стороны, для каждого из N элементов мы проводим операцию поиска, сложностью $\log N$, то есть сложность алгоритма должна быть $N * \log N$ (N раз по $\log N$). Но на самом деле, мы тратим время на вставку. В худшем случае это N операций копирования (если мы хотим что-то вставить в самое начало).

Но питон хорошо умеет работать со списками, поэтому на практике будет быстрее, если делать такие вещи не вручную.


### Алгоритмы логарифмической сложности

#### Сортировка слиянием

Этот алгоритм относится к алгоритмам «разделяй и властвуй».

Логика следующая:
1. Разбиваем список на две части, каждую из них разбиваем ещё на две и т. д. Список разбивается пополам, пока не останутся единичные элементы. Количество разбиений не превышает $\log_2 N$;
2. Соседние элементы становятся отсортированными парами (переставляем, если не так);
3. Затем эти пары объединяются и сортируются с другими парами;
4. Этот процесс продолжается до тех пор, пока не отсортируются все элементы.

Минусом алгоритма является тот факт, что каждый раз надо снимать копию с данных. На каждом шаге требуется N элементов памяти, всех шагов $\log_2 N$, итого требуется $N * \log_2 N$ элементов памяти. Это может быть достаточно много.

#### Быстрая сортировка

Этот алгоритм также относится к алгоритмам «разделяй и властвуй», но при правильной реализации он чрезвычайно эффективен и не требует дополнительной памяти, в отличие от сортировки слиянием.

Как это работает:
1. Массив разделяется на две части по разные стороны от опорного элемента;
2. В процессе сортировки элементы меньше опорного помещаются перед ним, а равные или большие — после.

Следует иметь в виду, что если нам не повезет с выбором опорного элемента, скорость сортировки будет равна $N^2$.

Рассмотрим этот алгоритм подробнее.

In [ ]:
# глобальный счётчик шагов
N_steps = 0


# Эта функция частично упорядочивает данные.
# Все меньше определенного элемента будут слева, остальные - справа от него.
def partition(nums, low, high):
    global N_steps

    # Выбираем средний элемент в качестве опорного
    # Также возможен выбор первого, последнего
    # или произвольного элементов в качестве опорного
    pivot = nums[(low + high) // 2]
    N_steps += 1  # выбор pivot

    i = low - 1
    j = high + 1

    while True:
        # Если элемент находится на своем месте надо просто сдвинуться.
        i += 1
        N_steps += 1  # движение i

        while nums[i] < pivot:
            i += 1
            N_steps += 1  # сравнение + движение

        # Если элемент находится на своем месте надо просто сдвинуться.
        j -= 1
        N_steps += 1  # движение j

        while nums[j] > pivot:
            j -= 1
            N_steps += 1  # сравнение + движение

        # Закончили перебор, встретились где-то посредине.
        if i >= j:
            N_steps += 1  # проверка завершения
            return j

        # Теперь точно известно, что слева и справа находятся элементы,
        # которые находятся не на своих местах. Поменяем их местами.
        nums[i], nums[j] = nums[j], nums[i]
        N_steps += 1  # swap


def quick_sort(nums):
    global N_steps
    N_steps = 0  # сброс счётчика

    nums = copy(nums)

    # Создадим вспомогательную функцию, которая вызывается рекурсивно
    def _quick_sort(items, low, high):
        global N_steps
        if low < high:
            N_steps += 1  # рекурсивный шаг
            split_index = partition(items, low, high)

            _quick_sort(items, low, split_index)
            _quick_sort(items, split_index + 1, high)

    _quick_sort(nums, 0, len(nums) - 1)

    return nums, N_steps

Потестируем

In [ ]:
N = 10000
lst = [random.randint(0, N) for i in range(N)]

In [ ]:
lst, n = quick_sort(lst)
print(n)
print(lst[10:30])

Теперь посмотрим на зависимость от длины:

In [ ]:
n_values = np.arange(10, 100) # иначе очень долго
steps = []

for n in n_values:
    lst = [random.randint(0, n) for i in range(n)]

    _, n_steps = quick_sort(lst)
    steps.append(n_steps)

plt.figure(figsize=(4, 6))
plt.plot(n_values, steps)

plt.xlabel("Array size")
plt.ylabel("N steps");

Заметим, что можно отсортировать массив и за линейное время. Достаточно посчитать частоты всех значений в списке с данными, а потом вывести все ненулевые значения соответствующее числов раз.

In [ ]:
data = [random.randint(0, 100) for _ in range(20)]

freqs = [0] * 101

for d in data:
    freqs[d] += 1

data3 = []
for i in range(101):
    data3.extend([i] * freqs[i])

In [ ]:
data3[10:40]

Быстрее, чем даже quick_sort! Но теперь представим себе, что мы сотрируем список жителей России или номера заграничных паспортов всех жителей планеты. Будет очень затруднительно завести список для всех возможных значений имен или номеров. В связи с этим метод очень редко применяется на практике.

## Индексирование

Иногда сортировка непосредственно данных может быть затруднена или невозможна. Например, если мы храним большой объем данных по каждому жителю Китая, Индии и США, то получается, что мы могли бы отсортировать такой объем чисел, но нам затруднительно отсортировать сами данные, так как манипуляции с такими объемами займут много времени.

В таком случае можно построить _индекс_.

Давайте создадим список, который будет хранить порядковые номера хранимых объектов. Дальше вместо того, чтобы обращаться к элементу с номером i, мы будем брать i-й элемент индекса и обращаться к элементу с полученным номером. Если считать, что данные хранятся в списке data, а индекс в списке index, то вместо обращения data[i] мы будем использовать data[index[i]].

Теперь вместо того, чтобы перемещать элемент в самом списке данных, мы будем изменять его номер в индексе. При сортировке это означает, что вместо перемещения элементов мы будем менять местами их индексы.

Пусть есть массив с числами [10, 15, 3, 5, 24, 6]. Построим индекс [0, 1, 2, 3, 4, 5]. Тогда после сортировки массив с данными не изменится, а индекс превратится в [2, 3, 5, 0, 1, 4]. Индекс будет показывать, в каком порядке следует взять элементы данных, для того, чтобы они шли в отсортированном виде.

Такой подход особенно удобен, когда нам надо иметь несколько сортировок. Например, у нас есть список собак нашего клуба. В зависимости от задачи, нам надо искать собак по числу медалей, кличке, адресу хозяина и т.д. Вместо того, чтобы сортировать каждый раз перед поиском, мы стрим несколько индексов, которые используем в зависимости от поставленной задачи.

Именно так работают базы данных.

Пусть у нас есть небольшая база людей:

In [ ]:
data = [
    {
        "id": i,
        "name": f"user_{i}",
        "age": random.randint(18, 60),
        "height": random.randint(150, 200)
    }
    for i in range(15)
]

data[0]

Мы хотим уметь быстро выводит сортировки по возрасту и росту. Для этого заведем индексы:

In [ ]:
age_index = [item['id'] for item in sorted(data, key=lambda x: x['age'])]
age_index

In [ ]:
height_index = [item['id'] for item in sorted(data, key=lambda x: x['height'])]
height_index

Выведем топ-5 по росту:

In [ ]:
n = 5
for idx in height_index[:-(n + 1):-1]:
    print(data[idx])

Правда, в Питоне мы не ощутим всей мощи этого подхода в смысле скорости, так как Питон перемещает не сами переменные, а ссылки на них. Фактически, уже использует этот подход.